## A Minimal Example of LCQP

In [ ]:
%load_ext autoreload
import numpy as np
from phisolve.solvers.phi_miqp import PhiMIQPParams, PhiMIQP
from phisolve.backends.commons import BackendParams
from phisolve.refiners.pdqp import PDQP
from phisolve.problems.lcqp import LCQP
import scipy

In [ ]:
seed = 42

np.random.seed(seed)

n = 20
m = 10
k = 3
U = scipy.stats.ortho_group(n, seed).rvs()
eig = np.random.normal(-10, 5, (n))
Q = U.T @ np.diag(eig) @ U
w = np.random.normal(0, 10, (n))
x = np.random.uniform(0, 1, (n))
A = np.random.normal(0, 1, (m, n))
b = np.random.normal(0, np.sqrt(n), (m))
infeas = A @ x > b
A[infeas] *= -1
b[infeas] *= -1
C = np.random.uniform(0, 1, (k, n))
d = C @ x
lcqp = LCQP(Q, w, A=A, b=b, C=C, d=d)

In [ ]:
n_shots = 100
n_steps = 10000
seed = 42
device = "cpu"

In [ ]:
import jax
from phisolve.utils.jax_utils import jax_device
jax.config.update("jax_platforms", jax_device(device))

In [ ]:
backend_params = BackendParams(n_shots=n_shots, n_steps=n_steps, seed=seed, device=device)
refiner = PDQP(device=device, iterations=10000).refine
solver_params = PhiMIQPParams(refine=refiner, backend_params=backend_params)

solver = PhiMIQP(lcqp)
res = solver.run(solver_params)

xs = res.refined_samples

objs = np.array([lcqp.obj(x) for x in xs])
maxvios = np.array([lcqp.max_vios(x) for x in xs])
feas = maxvios < 1e-4
minima = np.min(objs[feas])
minimizer = np.argmin(objs[feas])
minimizer_vios = maxvios[feas][minimizer]
print(minima, minimizer_vios, res.succ_prob(tol=1e-4))
assert minima < -127.39

In [ ]:
samples = np.random.normal(0, 1, (n_shots, lcqp.nvar))
pdqp = PDQP(problem=lcqp, device=device, iterations=10000)
res = pdqp.pdqp_main(samples)
objs = np.array([lcqp.obj(x) for x in res])
maxvios = np.array([lcqp.max_vios(x) for x in res])
feas = maxvios < 1e-4
minima = np.min(objs[feas])
minimizer = np.argmin(objs[feas])
succ = objs[feas] <= minima + 1e-4
minimizer_vios = maxvios[feas][minimizer]
succ_prob = np.sum(succ) / n_shots
print(minima, minimizer_vios, succ_prob)
assert minima < -127.39